In [1]:
import teehr
import pandas as pd
from teehr.evaluation.spark_session_utils import create_spark_session

from teehr import DeterministicMetrics as dm
from teehr import Signatures as s
from teehr import RowLevelCalculatedFields as rcf
from teehr import TimeseriesAwareCalculatedFields as tcf
from teehr import Bootstrappers as bs

from teehr.models.filters import TableFilter

from pyspark.sql import functions as F

from pyspark.sql import DataFrame

import copy
import time

teehr.__version__

'0.6.6'

In [2]:
spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


In [3]:
spark.sql("""USE iceberg""")
spark.sql("""
SELECT *
FROM teehr.nwmd_metrics_by_location_test
WHERE
LIMIT 10
""").show(10)

+-------------------+---------------------+------------------+---------+--------------------+------+-------+----------------------+---------+----------+-----+------------------+------------------+------------------+-------------------+-------------------+--------------------+-------------------+---------------------------+--------------------+-------------------------+----------------------+--------------------+------------------------+------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------------------+--------------------------------------+------------------------------------+------------------------------------+------------------------+------------------------+------------------------------+------------------------------+---------------------------------+---------------------------------+--------------------+----------------

In [4]:
spark.sql("""USE iceberg""")
spark.sql("""
SELECT *
FROM teehr.nwmd_metrics_by_location_test
WHERE primary_location_id = 'usgs-01094400'
    AND quarter = '2025-Q4'
    AND forecast_lead_time_bin = 'P1DT0H_P2DT0H'
    AND threshold = 'above_q85'
    AND window_agg = 'max'
ORDER BY primary_location_id
LIMIT 10
""").show()
spark.sql("""
SELECT *
FROM teehr.nwmd_metrics_by_location
WHERE primary_location_id = 'usgs-01094400'
    AND quarter = '2025-Q4'
    AND forecast_lead_time_bin = 'P1DT0H_P2DT0H'
    AND threshold = 'above_q85'
    AND window_agg = 'max'
ORDER BY primary_location_id
LIMIT 10
""").show()

+-------------------+---------------------+------------------+---------+--------------------+------+-------+----------------------+---------+----------+-----+------------------+-----------------+-----------------+-----------------+------------------+-------------------+------------------+---------------------------+-------------------+-------------------------+----------------------+-------------------+------------------------+------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------------------+--------------------------------------+------------------------------------+------------------------------------+------------------------+------------------------+------------------------------+------------------------------+---------------------------------+---------------------------------+--------------------+--------------------+----

In [11]:
spark.sql("SHOW TBLPROPERTIES teehr.nwmd_metrics_by_location").show(truncate=False)

+-------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|key                            |value                                                                                                                                                                                                                                                                                                                                                                                                                                                        |
+-------------------------------+-------

In [9]:
# Show field descriptions for the teehr.nwmd_metrics_by_location table
spark.sql("DESCRIBE EXTENDED teehr.primary_timeseries").show(truncate=False)

+------------------+-------------------------------------------------------------------------+------------------------------------------------------+
|col_name          |data_type                                                                |comment                                               |
+------------------+-------------------------------------------------------------------------+------------------------------------------------------+
|reference_time    |timestamp                                                                |NULL                                                  |
|value_time        |timestamp                                                                |NULL                                                  |
|value             |float                                                                    |NULL                                                  |
|unit_name         |string                                                                   |NULL  

## Full-table comparison: legacy (`nwmd_metrics_by_location`) vs. vectorized (`nwmd_metrics_by_location_test`)

Joins the two tables on the full uniqueness key set (`group_by`) and compares
every shared numeric metric column across *all* matching rows, rather than
spot-checking individual rows by hand. Requires the main pipeline's `spark`/
`ev`/`group_by` still be active (run this in the same session as the full
location-set run, before `spark.stop()`).

Expectations, per the legacy-vs-vectorized discussion above:
- Non-bootstrap columns should match (near) exactly -- they're computed by
  native Spark aggregation and were never touched by the bootstrap engine.
- Bootstrap quantile columns (`*_boot_0_025`/`*_boot_0_975`) should be close
  but not necessarily identical -- these two tables were written by separate
  Spark sessions, so row-order-dependent resampling can legitimately produce
  a different (equally valid) bootstrap draw of the same underlying data.


In [31]:
# --- Full-table comparison: legacy vs. vectorized, across every matching row ---
import pyspark.sql.functions as F
import pyspark.sql.types as T

legacy_sdf = ev.table("nwmd_metrics_by_location").to_sdf()
test_sdf = ev.table("nwmd_metrics_by_location_test").to_sdf()

join_keys = group_by  # same uniqueness fields both tables were written with

l = legacy_sdf.alias("l")
t = test_sdf.alias("t")

# Null-safe join -- member/threshold can be NULL, and a plain equality join
# would silently drop those rows (NULL <> NULL in standard SQL semantics).
join_cond = None
for k in join_keys:
    cond = l[k].eqNullSafe(t[k])
    join_cond = cond if join_cond is None else (join_cond & cond)

joined = l.join(t, on=join_cond, how="inner")

legacy_count = legacy_sdf.count()
test_count = test_sdf.count()
matched_count = joined.count()

print(f"legacy rows:  {legacy_count}")
print(f"test rows:    {test_count}")
print(f"matched rows: {matched_count}")
if matched_count < min(legacy_count, test_count):
    print(
        "NOTE: fewer matched rows than the smaller table -- the two tables may "
        "cover different location samples/date ranges (e.g. different quarters "
        "or a different location sample). Diffs below are only over rows "
        "present in both."
    )

# Compare only columns present (by name) in both tables, excluding join keys
# and non-numeric columns (e.g. geometry from add_geometry()).
numeric_types = (T.DoubleType, T.FloatType, T.LongType, T.IntegerType, T.DecimalType)
legacy_numeric_cols = {f.name for f in legacy_sdf.schema.fields if isinstance(f.dataType, numeric_types)}
test_numeric_cols = {f.name for f in test_sdf.schema.fields if isinstance(f.dataType, numeric_types)}
compare_cols = sorted((legacy_numeric_cols & test_numeric_cols) - set(join_keys))

agg_exprs = []
for c in compare_cols:
    diff = F.abs(F.col(f"l.{c}") - F.col(f"t.{c}"))
    rel = diff / F.greatest(F.abs(F.col(f"l.{c}")), F.lit(1e-12))
    agg_exprs.append(F.max(diff).alias(f"{c}__max_abs"))
    agg_exprs.append(F.max(rel).alias(f"{c}__max_rel"))
    agg_exprs.append(F.avg(rel).alias(f"{c}__mean_rel"))

summary = joined.agg(*agg_exprs).collect()[0].asDict()


def is_boot_col(c):
    return "_boot_" in c or c.endswith("_boot")


print(f"\n=== Non-bootstrap metrics (should match almost exactly) ===")
print(f"{'column':35s} {'max_abs_diff':>14s} {'max_rel_diff':>14s}")
for c in compare_cols:
    if not is_boot_col(c):
        print(f"{c:35s} {summary[f'{c}__max_abs']:14.3e} {summary[f'{c}__max_rel']:14.3e}")

print(f"\n=== Bootstrap metrics (close, not necessarily identical -- independent draws) ===")
print(f"{'column':35s} {'max_abs_diff':>14s} {'max_rel_diff':>14s} {'mean_rel_diff':>14s}")
for c in compare_cols:
    if is_boot_col(c):
        print(f"{c:35s} {summary[f'{c}__max_abs']:14.3e} {summary[f'{c}__max_rel']:14.3e} {summary[f'{c}__mean_rel']:14.3e}")


INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location.
INFO:teehr.evaluation.tables.generic_table:Getting table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: nwmd_metrics_by_location_test.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.nwmd_metrics_by_location_test.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.nwmd_metrics_by_location_test.


legacy rows:  3771336
test rows:    99108
matched rows: 99108

=== Non-bootstrap metrics (should match almost exactly) ===
column                                max_abs_diff   max_rel_diff
average                                  3.345e+00      9.687e-02
count                                    1.000e+00      5.000e-01
kling_gupta_efficiency                         nan            nan
maximum                                  5.947e+00      1.284e-01
minimum                                  1.274e+01      2.763e-01
nash_sutcliffe_efficiency                      nan            nan
pearson_correlation                      1.737e+00      1.000e+12
relative_bias                            9.685e-01      1.632e+02
relative_maximum                         2.725e+00      4.839e-01
relative_mean                            9.685e-01      3.497e-01
relative_median                          5.438e-01      4.150e-01
relative_minimum                         3.295e-01      4.522e+00
relative_standard_d

In [43]:
spark.stop()